# 2. Collective movement

Cells can bias their reorientation using neighboring occupancy.
This lesson compares random walk, polar alignment, aggregation and
nematic alignment using the same lattice, density and run length.

**Learning objectives**

- add different registered interactions to a visible ModelSpec;
- separate an interaction mechanism from its parameter strength;
- compare mechanisms with a quantitative observable; and
- distinguish polar and nematic organization.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    run_model,
)
from lgca.pipeline import InteractionPipelineSpec
from lgca.simulation import DensityRecorder, NodeRecorder, PopulationRecorder


## Build model variants, do not load them

The function below keeps experimental controls in one place, but
its body still shows the entire specification and the exact line
that inserts the interaction. This is useful when several models
differ by one named mechanism.


In [ ]:
def make_collective_spec(interaction, beta=2.0, seed=21):
    parameters = {} if interaction == "classical.random_walk" else {"beta": beta}
    return ModelSpec(
        description=Description(title=f"Collective movement: {interaction}"),
        space=SpaceSpec(
            geometry="hex",
            dims=(20, 20),
            boundary="periodic",
        ),
        state=StateSpec(
            density=0.2,
            restchannels=0,
        ),
        time=TimeSpec(
            steps=25,
            seed=seed,
        ),
        dynamics=InteractionPipelineSpec(
            operators=[{"name": interaction, "parameters": parameters}],
        ),
        analysis=AnalysisSpec(
            observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
        ),
    )


interaction_names = (
    "classical.random_walk",
    "classical.alignment",
    "classical.aggregation",
    "classical.nematic",
)


The four mechanisms answer different biological questions:

- **random walk:** what happens without a directional cue?
- **alignment:** can neighbor-induced orientation create streams?
- **aggregation:** does motion up a local density gradient form clusters?
- **nematic alignment:** can cells share an axis while moving in
  opposite directions along it?


In [ ]:
results = {
    name: run_model(make_collective_spec(name, beta=2.0, seed=21), showprogress=False)
    for name in interaction_names
}


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 7), constrained_layout=True)
for axis, (name, model_result) in zip(axes.flat, results.items()):
    image = axis.imshow(model_result.lgca.dens_t[-1].T, origin="lower")
    axis.set_title(name.removeprefix("classical.").replace("_", " "))
    axis.set_xlabel("x")
    axis.set_ylabel("y")
    fig.colorbar(image, ax=axis, shrink=0.75)
plt.show()
plt.close(fig)


## A global polarization observable

Visual differences are useful for exploration but insufficient for
comparisons. Global polarization measures the magnitude of the
summed velocity vector divided by particle number. It approaches
one when most particles move in one direction and stays small for
disordered motion. A nematic state can be strongly ordered yet have
low polarization because opposite directions cancel; it therefore
needs a nematic observable in a detailed study.


In [ ]:
def polarization(lgca, nodes):
    total_particles = nodes.sum()
    if total_particles == 0:
        return 0.0
    total_flux = lgca.calc_flux(nodes).sum(axis=(0, 1))
    return float(np.linalg.norm(total_flux) / total_particles)


final_polarization = {
    name: polarization(model_result.lgca, model_result.lgca.nodes_t[-1])
    for name, model_result in results.items()
}
final_polarization


In [ ]:
labels = [name.removeprefix("classical.").replace("_", " ") for name in interaction_names]
values = [final_polarization[name] for name in interaction_names]

fig, axis = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
axis.bar(labels, values)
axis.set_ylabel("final global polarization")
axis.tick_params(axis="x", rotation=25)
plt.show()
plt.close(fig)


## Interpretation and limitations

The controlled setup attributes differences to the interaction
rule, but a single seed and parameter value do not establish a
robust phase diagram. Aggregation is better quantified with a
clustering measure, and nematic alignment with an axis-sensitive
order parameter. Choosing an observable is part of choosing the
scientific question.

## Exercises

1. Sweep `beta` for alignment and plot polarization against beta.
2. Define a density-variance or occupied-cluster observable for
   aggregation. Does it agree with the density maps?
3. Construct a nematic order parameter that treats directions
   separated by 180 degrees as equivalent.
4. Repeat each mechanism for five seeds and add uncertainty bars.
